# 9 · When requirements change: evolving schemas

**The problem.** The CRM has been in production for a year. Now product wants:

1. a `pronouns` field on contacts, without breaking the thousands of saved snapshots,
2. a v2 contact with a different shape, running side by side with v1 during migration,
3. customer IDs that are unique across the whole system, and
4. contacts reachable by phone *or* email: a variant type.

Schemas are values, so changing a schema is ordinary code, with the same builders and the same `clone()` and
`update()` endings you use for instances. This last case study walks through each request and ends with what the
framework does *not* do yet.

In [1]:
from mbse.Schemas.Framework import JSON, Proxies, Schemas, Validators
from mbse.Schemas.Framework.Errors import DecodeError

store = Proxies.OfStore()

Contact = (
    Schemas.OfObject.Builder().name('crm.Contact').ref()
    .properties(
        lambda prop: prop.name('given_name').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('family_name').of(lambda t: t.as_native(str)),
    )
    .create()
)
store.register(Contact)   # registered names are strings: dots, versions, anything
NewContact = getattr(store, 'crm.Contact')  # names that aren't identifiers are reached with getattr

saved = JSON.ToJSON(store)(Contact, NewContact().given_name('Alice').family_name('Liddell').create())
saved

'{"root": "s0", "objects": {"s0": {"given_name": "Alice", "family_name": "Liddell"}}}'

## 1. Adding a field

`Builder(existing)...update()` changes the schema *in place*. Everything registered under that schema sees the change
at once, including existing instances and builders:

In [2]:
Schemas.OfObject.Builder(Contact).properties(lambda prop: prop.name('pronouns').of(lambda t: t.as_native(str))).update()

alice = JSON.FromJSON(store)(Contact, saved)   # a snapshot saved before the change
alice = NewContact(alice).pronouns('she/her').update()
print(alice.given_name, alice.pronouns)

Alice she/her


Old data loads without a migration step, because **every property is optional** (case study 1). A new property is
simply absent in old data. This is the payoff for not making fields mandatory by default.

The other direction is strict. A program still running the *old* schema refuses data carrying the new field, rather
than silently dropping it on the next save:

In [3]:
OldContact = (
    Schemas.OfObject.Builder().name('crm.OldContact').ref()
    .properties(
        lambda prop: prop.name('given_name').of(lambda t: t.as_native(str)),
        lambda prop: prop.name('family_name').of(lambda t: t.as_native(str)),
    )
    .create()
)
store.register(OldContact)
newer = JSON.ToJSON(store)(Contact, alice)
try:
    JSON.FromJSON(store)(OldContact, newer)
except DecodeError as error:
    print('DecodeError:', error)

DecodeError: $.objects.s0.pronouns: 'crm.OldContact' has no property or adjacency 'pronouns'


**Why refuse?** Loading is the last moment the program can notice that it doesn't understand the data. Dropping the
field would lose it on the next save, with nothing in any log. Deploy readers before writers, as with any strict
format.

## 2. A second version, side by side

`clone()` gives an independent copy to change. Register it under a new name, and v1 and v2 coexist:

In [4]:
ContactV2 = (
    Schemas.OfObject.Builder(Contact)
    .name('crm.Contact.v2')
    .properties(lambda prop: prop.name('display_name').of(lambda t: t.as_native(str)))
    .clone()
)
store.register(ContactV2)

print(list(Contact.properties))
print(list(ContactV2.properties))

['given_name', 'family_name', 'pronouns']
['given_name', 'family_name', 'pronouns', 'display_name']


A migration is ordinary code over the two builders:

In [5]:
def migrate(v1):
    return (
        getattr(store, 'crm.Contact.v2')()
        .given_name(v1.given_name)
        .family_name(v1.family_name)
        .display_name(f'{v1.given_name} {v1.family_name}')
        .create()
    )


v2 = migrate(alice)
print(v2.schema_name(), '|', v2.display_name)

crm.Contact.v2 | Alice Liddell


Because references carry schema names (case study 6), a snapshot that mixes v1 and v2 objects loads each with the
right schema.

Redefining a property replaces it, and that is how a type changes. `validate()` on the result, and on your data,
tells you what else needs to follow.

## 3. Unique customer IDs: a directory

"Every customer has an ID, and an ID names at most one customer" sounds like it needs a special feature. It's a
relation with `unique` clauses (case study 5), linking a directory object to each customer, with the ID as the entry
property:

In [6]:
Customers = (
    Schemas.OfRelation.Builder().name('crm.Customers')
    .links('directory', 'customer')
    .properties(lambda prop: prop.name('id').of(lambda t: t.as_native(str)))
    .unique('customer')   # (directory, id) determine the customer: an ID resolves to one customer
    .unique('id')         # (directory, customer) determine the id: a customer has one ID
    .create()
)
Directory = (
    Schemas.OfObject.Builder().name('crm.Directory').ref()
    .singleton('crm.Directory')  # declared as the one global directory
    .relations(lambda adj: adj.name('customers').of(Customers).me('directory'))
    .create()
)
Schemas.OfObject.Builder(ContactV2).relations(lambda adj: adj.name('ids').of(Customers).me('customer')).update()
store.register(Customers)
store.register(Directory)

directory = store.singleton('crm.Directory')  # made by the store, with its schema
bob = migrate(NewContact().given_name('Bob').family_name('Kane').create())
getattr(store, 'crm.Directory')(directory).customers(lambda x: x.customer(v2).id('C-1')).update()
getattr(store, 'crm.Directory')(directory).customers(lambda x: x.customer(bob).id('C-1')).update()  # oops

Validators.Validate(store).Reachable(Directory, directory)

["unique(customer) violated: entries agreeing on ['directory', 'id'] differ on ['customer']"]

Named references, maps, directories and ownership are all the same element: a relation plus `unique`. There's no
separate "ID" or "foreign key" feature to learn.

The `.singleton('crm.Directory')` declaration records that there's exactly one directory, referenced by its global
name. The store makes that instance when the schema is registered, and `store.singleton('crm.Directory')` gives it;
building another is refused. A store's singletons are its roots: the store keeps what they reach, here the directory and
every customer listed in it, and anything they don't reach is the program's to keep.

## 4. Variants: unions and intersections

"Reachable by phone or email" is a **union**: one of several same-kind schemas, each a named branch, and a value
holds exactly one of them. "Timestamped and audited" is an **intersection**: a value holding each of several named
parts. Each part keeps its own properties, so two parts may both declare `updated`, even with different types.

In [7]:
Phone = Schemas.OfObject.Builder().properties(lambda prop: prop.name('number').of(lambda t: t.as_native(str))).create()
Email = Schemas.OfObject.Builder().properties(lambda prop: prop.name('address').of(lambda t: t.as_native(str))).create()

Reach = (
    Schemas.OfUnion.Builder()
    .branches(lambda b: b.name('phone').of(Phone), lambda b: b.name('email').of(Email))
    .create()
)
print('union:', Reach.validate())

Timestamped = Schemas.OfObject.Builder().properties(lambda prop: prop.name('updated').of(lambda t: t.as_native(str))).create()
Epoch = Schemas.OfObject.Builder().properties(lambda prop: prop.name('updated').of(lambda t: t.as_native(int))).create()
Stamps = (
    Schemas.OfIntersection.Builder()
    .parts(lambda p: p.name('text').of(Timestamped), lambda p: p.name('epoch').of(Epoch))
    .create()
)
print('intersection:', Stamps.validate())

union: []
intersection: []


Unions and intersections can be declared and validated as schemas. A property whose schema is an object holds a
*value* object: a read-only value that belongs to its owner, is copied with it and is written nested in it. Union and intersection values
are written the same way, keyed by name: a union value `{"email": {"address": "..."}}` holds its one branch, and an
intersection value `{"text": {...}, "epoch": {...}}` each of its parts. Which branch a value holds is data, so
neither loading nor validation needs predicates, and reordering a union's branches leaves stored data valid.

A generated binding may store a union the way its language does (a tagged union, `std::variant`); the named form is
the neutral one that every program reads.

## What's designed but not built yet

The design document (`FRAMEWORK.md`) specifies more than the Python package implements today. It's worth knowing
where the edges are:

| Feature | Status |
|---|---|
| Expressions (constraints such as "at least one") | in the separate [mbse-expressions](https://github.com/pitaman71/mbse-expressions) package, which depends on this one |
| Mutations and transactions: changes as serializable data | designed, not implemented |
| Deleting objects (entries linking them vanish with them) | designed, not implemented |
| Generated bindings for other languages | planned, in separate repositories per target language; a hand-written TypeScript binding exists and matches this one |

Open questions are listed in `FRAMEWORK.md` under *Open questions*, several of them with a test that pins today's
behavior (for example F4, "last `update()` wins", which you met in case study 3).

## The whole tour

| Case study | Problem | Idea |
|---|---|---|
| 1 | a contact card | schemas are values; builders; `create`/`update`/`clone`; absent is not `None` |
| 2 | an address book | relations instead of lists; untyped links; entries are shared facts |
| 3 | moving house | editing entries through builders; what `update()` replaces |
| 4 | a family tree | self-relations, cycles, `Reachable.of`; unambiguous inference |
| 5 | a price list | `unique(...)` cardinality; validation on demand |
| 6 | saving and loading | snapshots, symbols, references with schema names |
| 7 | JSON and YAML | strict text; the Norway problem; no coercion |
| 8 | tools for every schema | walking schemas; writing a visitor |
| 9 | evolving schemas | optional fields; versions; directories; variants |

The through-line: **describe data once, as data, and let every tool read that description.** Most of the ways this
framework differs from the usual approach (no lists, no mandatory fields, no automatic validation, no type
coercion, no guessing) are there so that description means the same thing to every program that reads it.